# sdp-meta Retail Demo - Presenter Walkthrough

**sdp-meta** (Databricks Labs, `databricks-labs-sdp-meta==0.1.0`) is a metadata-driven framework for Lakeflow Spark Declarative Pipelines. You describe sources, targets, data quality and CDC in **YAML**; sdp-meta writes them into **dataflowspec** tables; one generic pipeline notebook reads the spec and builds the whole bronze/silver graph.

| Layer | Schema | Built by |
| --- | --- | --- |
| Landing files | `sdp_meta_landing.files` (volume) | `00_setup_landing`, `01_simulate_increments` |
| Control | `sdp_meta_control` | onboarding job (dataflowspec tables) |
| Bronze / Silver | `sdp_meta_bronze`, `sdp_meta_silver` | 4 sdp-meta pipelines (2 data_flow_groups x 2 layers) |
| Gold | `sdp_meta_gold` | SQL pipeline `gold_retail` + 3 metric views |

Catalog: `mg_aws_serverless_workspace_catalog`. Everything is serverless and deployed with Declarative Automation Bundles.

All cells below are **read-only**.

## 1. Metadata in, pipelines out

Show in the repo:
* `conf/onboarding.yml` - one entry per data flow (ids 101-105 retail_reference, 201-203 retail_sales)
* `conf/dqe/<entity>/bronze_expectations.yml` - `expect`, `expect_or_drop`, `expect_or_quarantine`, `expect_or_fail`
* `conf/silver_transformations/<group>.yml` - `select_exp` / `where_clause`
* `notebooks/init_sdp_meta_pipeline` - the *only* pipeline code (a few lines)

The onboarding job turns YAML into rows:

In [0]:
%sql
SELECT
  'bronze' AS layer, dataFlowId, dataFlowGroup, sourceFormat,
  sourceDetails['path'] AS source_path,
  targetDetails['table'] AS target_table,
  dataQualityExpectations IS NOT NULL AS has_dqe,
  version, updateDate
FROM mg_aws_serverless_workspace_catalog.sdp_meta_control.bronze_dataflowspec
UNION ALL
SELECT
  'silver', dataFlowId, dataFlowGroup, sourceFormat,
  CONCAT_WS('.', sourceDetails['database'], sourceDetails['table']),
  targetDetails['table'],
  dataQualityExpectations IS NOT NULL,
  version, updateDate
FROM mg_aws_serverless_workspace_catalog.sdp_meta_control.silver_dataflowspec
ORDER BY layer, dataFlowId

## 2. Auto Loader, multi-format, reader options

Every bronze table is an Auto Loader stream (`cloudFiles`) - formats differ only in metadata:
* products: CSV with `;` delimiter
* promotions: CSV with `multiLine` + `escape`
* active_promotions: Parquet
* sales_orders / sales_stream: JSON

Open the bronze pipelines to show the generated graph (no per-table code).

In [0]:
%sql
SELECT table_name, table_type, comment
FROM mg_aws_serverless_workspace_catalog.information_schema.tables
WHERE table_schema IN ('sdp_meta_bronze', 'sdp_meta_silver', 'sdp_meta_gold')
ORDER BY table_schema, table_name

## 3. Data quality expectations and quarantine

* `expect` - warn only (e.g. known loyalty segment -> 1 customer kept with segment 9 on purpose)
* `expect_or_drop` - bad rows removed from bronze
* `expect_or_quarantine` - bad rows routed to `<table>_quarantine`
* `expect_or_fail` - `sales_orders.order_number IS NOT NULL` stops the update (demoed in batch 4)

Dashboard page 3 shows the same numbers from the event logs.

In [0]:
%sql
SELECT 'products' AS entity,
  (SELECT COUNT(*) FROM mg_aws_serverless_workspace_catalog.sdp_meta_bronze.products) AS loaded,
  (SELECT COUNT(*) FROM mg_aws_serverless_workspace_catalog.sdp_meta_bronze.products_quarantine) AS quarantined
UNION ALL
SELECT 'customers',
  (SELECT COUNT(*) FROM mg_aws_serverless_workspace_catalog.sdp_meta_bronze.customers),
  (SELECT COUNT(*) FROM mg_aws_serverless_workspace_catalog.sdp_meta_bronze.customers_quarantine)
UNION ALL
SELECT 'sales_orders',
  (SELECT COUNT(*) FROM mg_aws_serverless_workspace_catalog.sdp_meta_bronze.sales_orders),
  (SELECT COUNT(*) FROM mg_aws_serverless_workspace_catalog.sdp_meta_bronze.sales_orders_quarantine)
UNION ALL
SELECT 'sales_stream',
  (SELECT COUNT(*) FROM mg_aws_serverless_workspace_catalog.sdp_meta_bronze.sales_stream),
  (SELECT COUNT(*) FROM mg_aws_serverless_workspace_catalog.sdp_meta_bronze.sales_stream_quarantine)

In [0]:
%sql
SELECT *
FROM mg_aws_serverless_workspace_catalog.sdp_meta_bronze.sales_orders_quarantine
LIMIT 10

## 4. SCD Type 2 with `silver_cdc_apply_changes`

Customers use AUTO CDC with `scd_type: 2`, keyed on `customer_id`, sequenced by `operation_ts`, with DELETE handling on `operation`. Batches 2-3 of `01_simulate_increments` added updates and deletes - history is preserved:

In [0]:
%sql
WITH multi AS (
  SELECT customer_id
  FROM mg_aws_serverless_workspace_catalog.sdp_meta_silver.customers
  GROUP BY customer_id
  HAVING COUNT(*) > 1
  LIMIT 3
)
SELECT c.*
FROM mg_aws_serverless_workspace_catalog.sdp_meta_silver.customers c
JOIN multi USING (customer_id)
ORDER BY customer_id, __START_AT

## 5. Multi-source CDC into one silver table

Data flow 203 (silver only) uses `silver_cdc_apply_changes_flows` to merge **batch** `sales_orders` and **streaming** `sales_stream` into a single `silver.orders` (SCD1 on `order_number`). No custom code - just two flow entries in the onboarding YAML.

In [0]:
%sql
SELECT `Source Channel`, MEASURE(`Orders`) AS orders, MEASURE(`Net Revenue`) AS net_revenue
FROM mg_aws_serverless_workspace_catalog.sdp_meta_gold.retail_sales_metrics
GROUP BY ALL

## 6. data_flow_group, versioning and metadata-only onboarding

* **data_flow_group** - `retail_reference` and `retail_sales` run as separate pipelines from the same spec tables (independent schedules / file-arrival triggers).
* **Versioning** - every onboarding run writes `version`, `createdBy`, `updateDate`; dataflowspec tables are Delta, so `DESCRIBE HISTORY` gives a full audit trail.
* **Adding a table = adding YAML** - uncomment `suppliers` in `conf/onboarding.yml`, rerun the job, and a new bronze/silver table appears with zero code changes.

In [0]:
%sql
DESCRIBE HISTORY mg_aws_serverless_workspace_catalog.sdp_meta_control.bronze_dataflowspec

## 7. Gold, metric views and dashboard

* `gold/gold_retail.sql` - SQL pipeline: `fact_order_lines`, `dim_customer_current`, `customer_history_changes`
* `gold/metric_views/*.yml` - `retail_sales_metrics`, `customer_metrics`, `customer_change_metrics` (governed measures, queried with `MEASURE()`)
* Dashboard **SDP Meta Retail Demo** - Sales & Promotions, Customers & Loyalty, Data Quality & Pipeline Health

## 8. Orchestration

Job `sdp_meta_retail_demo`: onboard -> bronze x2 -> silver x2 -> gold -> metric_views (all serverless, defined in `resources/sdp_meta_retail_demo_job.yml`).

In [0]:
%sql
SELECT `Change Type`, MEASURE(`Change Events`) AS events, MEASURE(`Customers Affected`) AS customers
FROM mg_aws_serverless_workspace_catalog.sdp_meta_gold.customer_change_metrics
GROUP BY ALL